# Analog Holidays

Thin orchestration notebook for running `AnalogSpecialDays` from the hourly wide holiday audit CSV.

The loading, normalization, forecasting, and plotting logic lives in `analog/analog_holidays.py`. This notebook only defines parameters and calls the plotting helpers.

The CSV export only preserves holiday flags, so this workflow targets holiday analogs rather than manual `special_day` or outlier labels.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.analog.analog_special_days as analog_special_days_module
import analog_holidays.analog.analog_holidays as analog_holidays_module

analog_special_days_module = importlib.reload(analog_special_days_module)
analog_holidays_module = importlib.reload(analog_holidays_module)

from analog_holidays.analog.analog_holidays import (
    build_analog_ranking_table,
    build_run_summary,
    plot_analog_pair_sequences,
    plot_batch_inference_grid,
    plot_batch_pair_sequences_grid,
    plot_forecast_diagnostics,
    plot_ranked_analog_profiles,
    run_analog_holidays,
    run_analog_holidays_batch,
    tune_analog_holidays_optuna,
)

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 20)

## Parameters

Adjust the series, target date, and analog hyperparameters for the holiday-only CSV source.

- SOURCE_PATH: path to the holiday CSV file consumed by the notebook.
- UNIQUE_ID: target series used to build analogs and generate the forecast.
- TARGET_DATE: target date for which the hourly profile should be estimated.
- SPECIAL_LABELS: labels that define which days are treated as special candidates when selecting analogs.
- SEASON_LENGTH: hourly profile length to model, in hours.
- K: number of special neighbors kept after ranking X against Y by similarity; None uses every filtered candidate.
- TYPEDIST: metric used to rank holiday candidates against Y; supports pearson, euclidian, and dtw.
- TYPEREG: regressor type used in the analog reconstruction step.
- N_COMPONENTS: number of components for dimensionality-reduction methods such as PCR or PLS.
- LEVELS: prediction interval levels to compute for the forecast.
- MIN_SPECIAL_POINTS: minimum number of hours flagged as special inside a candidate block.
- MIN_EVENT_GAP: minimum separation between consecutive special events to avoid overly overlapping candidates.
- MAX_EVENTS: maximum number of special events used as the analog bank; None uses all available events.
- MAX_PLOTTED_ANALOGS: maximum number of analogs shown in the comparative plots.

In [ ]:
SOURCE_PATH = PROJECT_ROOT / 'analog_holidays' / 'holidays' / 'holiday_demand_mx.csv'

UNIQUE_ID = 'SEN_demand_CEL'
UNIQUE_ID = 'SEN_demand_PEN'
UNIQUE_ID = 'SEN_demand_SIN'

In [ ]:
SPECIAL_LABELS = ('holiday',)
SEASON_LENGTH = 24
K = 1000
TYPEDIST = 'pearson'
TYPEREG = 'PCR'
N_COMPONENTS = 3
LEVELS = [80, 95]
MIN_SPECIAL_POINTS = 24
MIN_EVENT_GAP = 24
MAX_EVENTS = None
MAX_PLOTTED_ANALOGS = 10

DATE_END = '2024-01-01'  # only dates strictly earlier than this cutoff are included in the study
OPTUNA_N_TRIALS = 25
OPTUNA_TIMEOUT_SEC = 900
OPTUNA_MAX_EVAL_DATES = 12
OPTUNA_RANDOM_SEED = 42

In [ ]:
# TARGET_DATES_2025 = [
#     ('2024-01-01', "New Year's Day"),
#     ('2024-02-05', 'Constitution Day'),
#     ('2024-03-18', "Benito Juarez's Birthday"),
#     ('2024-03-28', 'Maundy Thursday'),
#     ('2024-03-29', 'Good Friday'),
#     ('2024-03-30', 'Holy Saturday'),
#     ('2024-05-01', 'Labor Day'),
#     ('2024-09-15', 'Independence Eve'),
#     ('2024-10-01', 'Presidential Inauguration'),
#     ('2024-11-02', 'Day of the Dead'),
#     ('2024-11-18', 'Mexican Revolution Day'),
#     ('2024-12-24', 'Christmas Eve'),
#     ('2024-12-25', 'Christmas Day'),
#     ('2024-12-31', "New Year's Eve"),
      # ===== 2025 =====
TARGET_DATES_2025 = [
    ('2025-01-01', "New Year's Day"),
    ('2025-02-03', 'Constitution Day'),
    ('2025-03-17', "Benito Juarez's Birthday"),
    ('2025-04-17', 'Maundy Thursday'),
    ('2025-04-18', 'Good Friday'),
    ('2025-04-19', 'Holy Saturday'),
    ('2025-05-01', 'Labor Day'),
    ('2025-09-15', 'Independence Eve'),
    ('2025-09-16', 'Independence Day'),
    ('2025-11-17', 'Mexican Revolution Day'),
    ('2025-12-24', 'Christmas Eve'),
    ('2025-12-25', 'Christmas Day'),
    ('2025-12-31', "New Year's Eve"),
    # ===== 2026 =====
    ('2026-01-01', "New Year's Day"),    
    ('2026-02-02', 'Constitution Day'),
    ('2026-03-16', "Benito Juarez's Birthday"),
    ('2026-04-02', 'Maundy Thursday'),
    ('2026-04-03', 'Good Friday'),
    ('2026-04-04', 'Holy Saturday'),
    ('2026-05-01', 'Labor Day'),
    # ('2026-09-15', 'Independence Eve'),
    # ('2026-09-16', 'Independence Day'),
    # ('2026-11-16', 'Mexican Revolution Day'),
    # ('2026-12-24', 'Christmas Eve'),
    # ('2026-12-25', 'Christmas Day'),
    # ('2026-12-31', "New Year's Eve"),
]

TARGET_DATE = TARGET_DATES_2025[-1][0]

In [ ]:
optuna_result = tune_analog_holidays_optuna(
    unique_id=UNIQUE_ID,
    source_path=SOURCE_PATH,
    train_end=DATE_END,
    season_length=SEASON_LENGTH,
    initial_k=K,
    initial_typedist=TYPEDIST,
    initial_typereg=TYPEREG,
    initial_n_components=N_COMPONENTS,
    n_trials=OPTUNA_N_TRIALS,
    timeout_sec=OPTUNA_TIMEOUT_SEC,
    max_eval_dates=OPTUNA_MAX_EVAL_DATES,
    random_seed=OPTUNA_RANDOM_SEED,
    special_labels=SPECIAL_LABELS,
    min_special_points=MIN_SPECIAL_POINTS,
    min_event_gap=MIN_EVENT_GAP,
    max_events=MAX_EVENTS,
 )

K = int(optuna_result.best_config['k'])
TYPEDIST = optuna_result.best_config['typedist']
TYPEREG = optuna_result.best_config['typereg']
N_COMPONENTS = int(optuna_result.best_config['n_components'])

print('Hyperparameters updated for the next notebook run.')
display(optuna_result.summary_df)
optuna_result.fold_metrics_df

In [ ]:
batch_result_2025 = run_analog_holidays_batch(
    target_dates=TARGET_DATES_2025,
    unique_id=UNIQUE_ID,
    source_path=SOURCE_PATH,
    season_length=SEASON_LENGTH,
    k=K,
    typedist=TYPEDIST,
    typereg=TYPEREG,
    n_components=N_COMPONENTS,
    levels=LEVELS,
    special_labels=SPECIAL_LABELS,
    min_special_points=MIN_SPECIAL_POINTS,
    min_event_gap=MIN_EVENT_GAP,
    max_events=MAX_EVENTS,
    expected_target_label=None,
 )

display(batch_result_2025.results_df)
batch_result_2025.metric_summary_df

In [ ]:
fig, axes = plot_batch_inference_grid(
    batch_result_2025,
    title=f'Batch inference | {UNIQUE_ID}\n{TYPEREG} | {TYPEDIST} | k={K}',
)
plt.show()

In [ ]:
run = batch_result_2025.runs.get(TARGET_DATE)

if run is None:
    run = run_analog_holidays(
        unique_id=UNIQUE_ID,
        target_date=TARGET_DATE,
        source_path=SOURCE_PATH,
        season_length=SEASON_LENGTH,
        k=K,
        typedist=TYPEDIST,
        typereg=TYPEREG,
        n_components=N_COMPONENTS,
        levels=LEVELS,
        special_labels=SPECIAL_LABELS,
        min_special_points=MIN_SPECIAL_POINTS,
        min_event_gap=MIN_EVENT_GAP,
        max_events=MAX_EVENTS,
    )

summary_df = build_run_summary(run)
print(summary_df.to_string(index=False))

In [ ]:
diagnostic_run = run_analog_holidays(
    unique_id=UNIQUE_ID,
    target_date=TARGET_DATE,
    source_path=SOURCE_PATH,
    season_length=SEASON_LENGTH,
    k=K,
    typedist=TYPEDIST,
    typereg=TYPEREG,
    n_components=N_COMPONENTS,
    levels=LEVELS,
    special_labels=SPECIAL_LABELS,
    min_special_points=MIN_SPECIAL_POINTS,
    min_event_gap=MIN_EVENT_GAP,
    max_events=MAX_EVENTS,
    expected_target_label=None,
)

interval_rows = []
for lv in LEVELS:
    lo = diagnostic_run.interval_low.get(lv)
    hi = diagnostic_run.interval_high.get(lv)
    if lo is None or hi is None:
        continue
    width = hi - lo
    interval_rows.append({
        'level': lv,
        'average_interval_width': float(np.mean(width)),
        'max_interval_width': float(np.max(width)),
        'min_interval_width': float(np.min(width)),
    })

display(pd.DataFrame(interval_rows))

level_to_show = 95 if 95 in LEVELS else max(LEVELS)
hourly_interval_df = pd.DataFrame({
    'hour': np.arange(1, len(diagnostic_run.forecast_profile) + 1),
    'forecast_mean': diagnostic_run.forecast_profile,
    f'lower_limit_{level_to_show}': diagnostic_run.interval_low[level_to_show],
    f'upper_limit_{level_to_show}': diagnostic_run.interval_high[level_to_show],
})

hourly_interval_df.head(10)

### X/X2 and Y/Y2 Sequences — Batch Grid

Una gráfica por fecha pronosticada, mostrando los pares históricos X/X2 en azul claro y la secuencia Y/Y2 forecast en rojo.


In [ ]:
fig_seq, axes_seq = plot_batch_pair_sequences_grid(batch_result_2025)
plt.show()


In [ ]:
ranking_df = build_analog_ranking_table(run)
fig, axes = plot_ranked_analog_profiles(run, top_n=MAX_PLOTTED_ANALOGS)
plt.show()
ranking_df.head(MAX_PLOTTED_ANALOGS)